# Phase 2C — Load flow

Run pandapower load flow on the network assembled by `13_pandapower_build.ipynb`, three scenarios. Newton–Raphson where it converges; DC fallback where it doesn't.

**Scenarios** (load + generator scaling factors applied uniformly):

| Scenario | Load × | Note |
|---|---:|---|
| `off_peak` | 0.40 | Lowest demand, NR converges cleanly |
| `morning_peak` | 0.85 | NR diverges at this loading on the current Phase 1 network — DC fallback |
| `evening_peak` | 0.95 | Same — DC fallback |

**Why NR diverges at heavy load.** Phase 1's synthetic 13.8 kV radial distribution feeders carry `r=0.4, x=0.4 Ω/km`. Cumulative voltage drop on long feeders pushes `vm_pu` below 0.5 at the network extremities, where the Jacobian becomes ill-conditioned. The DC approximation skips the voltage equation entirely, so it converges — at the cost of losing `vm_pu` / `va_degree` / reactive-power results. Phase 2D documents this as a Phase 1 distribution-impedance calibration concern.

**Output:** `backend/data/processed/load_flow_results.csv` — one row per (scenario, bus) and one per (scenario, line / transformer). Schema matches `backend/db/init.sql` so a PostGIS load is mechanical once Docker is available.

In [1]:
from pathlib import Path
import time
import pandas as pd
import pandapower as pp

PROC = Path('../backend/data/processed')
BASE_NET = PROC / 'pp_network.json'

SCENARIOS = [
    ('off_peak',      0.40),
    ('morning_peak',  0.85),
    ('evening_peak',  0.95),
]

# Generators stay at the build-time dispatch; the ext_grid slack absorbs the
# difference (negative slack import at low load = net export to Luzon).
SCALE_GEN_WITH_LOAD = False

## §1 — Run scenarios

In [2]:
def scaled_net(factor: float):
    n = pp.from_json(BASE_NET)
    n.load['p_mw'] = n.load['p_mw'] * factor
    n.load['q_mvar'] = n.load['q_mvar'] * factor
    if SCALE_GEN_WITH_LOAD:
        n.gen['p_mw'] = n.gen['p_mw'] * factor
    return n

summary = []
results = {}  # scenario -> (net, mode)
for name, factor in SCENARIOS:
    net = scaled_net(factor)
    in_load_mw = float(net.load.loc[net.load['in_service'], 'p_mw'].sum())
    gen_mw = float(net.gen['p_mw'].sum())
    t0 = time.time()
    mode = None; conv_iters = None
    try:
        pp.runpp(net, algorithm='nr', init='dc', numba=False,
                 calculate_voltage_angles=True, max_iteration=50)
        mode = 'nr'
    except pp.powerflow.LoadflowNotConverged:
        net = scaled_net(factor)
        pp.rundcpp(net, numba=False)
        mode = 'dc'
    dt = time.time() - t0

    results[name] = (net, mode)
    rb = net.res_bus.dropna(subset=['vm_pu' if mode == 'nr' else 'va_degree'])
    rl = net.res_line.dropna(subset=['loading_percent'])
    rt = net.res_trafo.dropna(subset=['loading_percent'])
    slack_p = float(net.res_ext_grid['p_mw'].iloc[0])
    summary.append({
        'scenario': name,
        'factor': factor,
        'mode': mode,
        'wall_s': round(dt, 2),
        'load_mw_in_svc': round(in_load_mw, 1),
        'gen_mw': round(gen_mw, 1),
        'slack_p_mw': round(slack_p, 1),
        'vm_pu_min': round(rb['vm_pu'].min(), 3) if mode == 'nr' else None,
        'vm_pu_max': round(rb['vm_pu'].max(), 3) if mode == 'nr' else None,
        'line_load_max_pct': round(rl['loading_percent'].max(), 1),
        'trafo_load_max_pct': round(rt['loading_percent'].max(), 1) if len(rt) else None,
    })

pd.DataFrame(summary)

,scenario,factor,mode,wall_s,load_mw_in_svc,gen_mw,slack_p_mw,vm_pu_min,vm_pu_max,line_load_max_pct,trafo_load_max_pct
0,off_peak,0.40,nr,0.03,328.1,600.0,-243.3,0.903,1.02,180.1,81.9
1,morning_peak,0.85,nr,0.02,697.2,600.0,134.4,0.779,1.02,190.2,80.2
2,evening_peak,0.95,nr,0.02,779.2,600.0,222.5,0.743,1.02,195.3,81.4


## §2 — Assemble unified results table

Schema matches `backend/db/init.sql.load_flow_results` exactly so the eventual PostGIS load is a `COPY FROM`. One row per (scenario, bus) and one per (scenario, line / trafo). For bus rows `bus_id` is set and `line_id` is null; for line/trafo rows the inverse. NR-only fields (`vm_pu`, `va_degree`) are blank for DC scenarios.

In [3]:
all_rows = []
for scen, (net, mode) in results.items():
    rb = net.res_bus.copy()
    rb['bus_id'] = net.bus['name']
    rb['in_service'] = net.bus['in_service']
    rb = rb[rb['in_service']].copy()
    for r in rb.itertuples():
        all_rows.append({
            'scenario': scen,
            'bus_id': r.bus_id,
            'line_id': None,
            'vm_pu': r.vm_pu if mode == 'nr' else None,
            'va_degree': r.va_degree,
            'loading_percent': None,
            'p_from_mw': None,
            'p_to_mw': None,
            'convergence_mode': mode,
        })

    for table_name, res_df, src_df in [('line', net.res_line, net.line),
                                       ('trafo', net.res_trafo, net.trafo)]:
        r2 = res_df.copy()
        r2['line_id'] = src_df['name'].values
        r2['in_service'] = src_df['in_service'].values
        r2 = r2[r2['in_service']].copy()
        for r in r2.itertuples():
            p_from = getattr(r, 'p_from_mw', None) if table_name == 'line' else getattr(r, 'p_hv_mw', None)
            p_to = getattr(r, 'p_to_mw', None) if table_name == 'line' else getattr(r, 'p_lv_mw', None)
            all_rows.append({
                'scenario': scen,
                'bus_id': None,
                'line_id': r.line_id,
                'vm_pu': None,
                'va_degree': None,
                'loading_percent': r.loading_percent,
                'p_from_mw': p_from,
                'p_to_mw': p_to,
                'convergence_mode': mode,
            })

df = pd.DataFrame(all_rows)
print(f'Total rows: {len(df)}')
print(df.groupby(['scenario', 'convergence_mode']).agg(
    rows=('scenario', 'size'),
    bus_rows=('bus_id', lambda s: s.notna().sum()),
    edge_rows=('line_id', lambda s: s.notna().sum()),
).to_string())

Total rows: 7536
                               rows  bus_rows  edge_rows
scenario     convergence_mode                           
evening_peak nr                2512      1224       1288
morning_peak nr                2512      1224       1288
off_peak     nr                2512      1224       1288


## §3 — Save

In [4]:
out = PROC / 'load_flow_results.csv'
df.to_csv(out, index=False)
print(f'Wrote {out} ({len(df)} rows, {out.stat().st_size / 1024:.0f} KB)')

out_sum = PROC / 'load_flow_summary.csv'
pd.DataFrame(summary).to_csv(out_sum, index=False)
print(f'Wrote {out_sum} ({len(summary)} scenarios)')

Wrote ../backend/data/processed/load_flow_results.csv (7536 rows, 669 KB)
Wrote ../backend/data/processed/load_flow_summary.csv (3 scenarios)
